# 02 KIỂM THỬ TRẠNG THÁI TRÒ CHƠI (GAME STATE TESTING)

Notebook này thực hiện kiểm thử toàn diện module `commander_game_engine.game_state` và `commander_game_engine.initial_setup`:

1. **Mô hình Dữ liệu Quân cờ (`Piece`, `PieceType`, `Player`)**:
   - Kiểm tra định nghĩa 11 loại quân và 2 phe đối kháng (Đỏ và Xanh).
   - Kiểm tra các thuộc tính thực thể: vị trí, quan hệ chuyên chở (`carried_by`, `cargo`), và trạng thái anh hùng (`is_heroic`).
2. **Khởi tạo Trạng thái Bàn cờ (`GameState` & `create_initial_state`)**:
   - Khởi tạo trạng thái mặc định rỗng.
   - Khởi tạo trạng thái ban đầu chuẩn gồm 38 quân cờ (19 quân Đỏ, 19 quân Xanh) theo đúng quy chuẩn Cờ Tư Lệnh.
3. **Các Phương thức Truy xuất Dữ liệu (Query & Access Methods)**:
   - `get_piece(piece_id)`: Tìm kiếm quân theo định danh ID.
   - `get_piece_at(pos)`: Tìm quân trực tiếp tại giao điểm và kiểm tra bắt lỗi ngoại lệ ngoài biên bàn cờ.
   - `get_pieces_by_player(player)`: Truy xuất danh sách quân hiện còn theo từng phe.
4. **Kiểm soát Ngoại lệ khi Thao tác Thêm quân (`add_piece`)**:
   - Kiểm tra 6 trường hợp ngoại lệ chặt chẽ: trùng ID, thiếu tọa độ, thêm trực tiếp quân đang được chở, cargo chưa giải quyết, tọa độ ngoài biên, ô đã có quân.
5. **Tính Bất biến & Độc lập của Phép Sao chép Sâu (`copy()` / Deepcopy)**:
   - Xác minh `state.copy()` tạo bản sao độc lập hoàn toàn trên bộ nhớ (`id` tách biệt).
   - Đảm bảo đột biến trên bản sao không gây tác dụng phụ (side-effects) lên trạng thái gốc.
6. **Bảo vệ Bất biến Cấu trúc & Quan hệ Chuyên chở (`validate_structure()`)**:
   - Kiểm tra tính nhất quán 2 chiều giữa phương tiện chở (`carrier`) và quân được chở (`cargo`).
   - Phát hiện xung đột phe giữa phương tiện và hàng chở.
   - Phát hiện chu trình chuyên chở vòng lặp (Circular transport loop detection).
   - Kiểm tra tính hợp lệ của lượt ghép (`CompoundTurnState`).
   - Kiểm tra các ràng buộc về điểm số (`scores`), thời gian (`elapsed_time`), lượt (`turn_number`), và số hành động (`action_count`).
7. **Thống kê Trạng thái & Trực quan hóa Phân bố Quân cờ**:
   - Bảng tổng hợp số lượng 11 loại quân cờ cho cả 2 phe.
   - Trực quan hóa ma trận chiếm giữ bàn cờ $12 \times 11$ (132 giao điểm).
8. **Kết luận Kiểm thử GameState**:
   - Đánh giá mức độ toàn vẹn, hiệu năng và sự sẵn sàng cho các thuật toán tìm kiếm Minimax, Alpha-Beta và MCTS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from copy import deepcopy

# Cấu hình nạp module
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "packages").is_dir() and (p / "research").is_dir()), None)
if str(PROJECT_ROOT / "packages") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "packages"))

from commander_game_engine.board import Board, Position, Terrain
from commander_game_engine.game_state import (
    GameState,
    Piece,
    PieceType,
    Player,
    GameStatus,
    CompoundTurnState
)
from commander_game_engine.initial_setup import (
    create_initial_state,
    INITIAL_POSITIONS,
    EXPECTED_COUNTS
)

print("Nạp module packages/commander_game_engine/game_state thành công!")
print(f"Phiên bản Python: {sys.version.split()[0]}")
print(f"Thư mục gốc dự án: {PROJECT_ROOT}")

Nạp module packages/commander_game_engine/game_state thành công!
Phiên bản Python: 3.10.0
Thư mục gốc dự án: D:\Commander-Chess-AI


## 1. Mô hình Dữ liệu Quân cờ (`Piece`, `PieceType`, `Player`)

Mỗi thực thể quân cờ trong Cờ Tư Lệnh được định danh bởi:
- `id`: Định danh duy nhất (ví dụ: `red_commander_1`, `blue_tank_2`).
- `piece_type`: Thuộc 1 trong 11 loại quân chuẩn của Cờ Tư Lệnh.
- `owner`: Phe sở hữu (`Player.RED` hoặc `Player.BLUE`).
- `position`: Tọa độ $(row, col)$ nếu đang đứng trên bàn cờ, hoặc `None` nếu đang được chở trên phương tiện.
- `carried_by` & `cargo`: Quan hệ chuyên chở liên binh chủng (Hải quân chở Xe tăng/Bộ binh, Máy bay chở Bộ binh...).
- `is_heroic`: Trạng thái quân anh hùng.

In [2]:
# 1.1. Kiểm tra 11 loại quân cờ
expected_piece_types = [
    PieceType.COMMANDER, PieceType.INFANTRY, PieceType.MILITIA,
    PieceType.TANK, PieceType.ARTILLERY, PieceType.ANTI_AIRCRAFT,
    PieceType.MISSILE, PieceType.AIRCRAFT, PieceType.NAVY,
    PieceType.ENGINEER, PieceType.HEADQUARTERS
]
assert len(PieceType) == 11, f"Kỳ vọng 11 loại quân, thực tế có {len(PieceType)}"
print(f"PASS: Đã xác thực đủ 11 loại quân cờ: {[pt.value for pt in PieceType]}")

# 1.2. Kiểm tra khởi tạo thực thể quân cờ đơn vị
p_cmd = Piece(id="red_commander_1", piece_type=PieceType.COMMANDER, owner=Player.RED, position=(0, 6))
assert p_cmd.id == "red_commander_1"
assert p_cmd.piece_type == PieceType.COMMANDER
assert p_cmd.owner == Player.RED
assert p_cmd.position == (0, 6)
assert p_cmd.carried_by is None
assert p_cmd.cargo == []
assert p_cmd.is_heroic is False
print("PASS: Khởi tạo quân cờ tiêu chuẩn thành công!")

# 1.3. Kiểm tra quân cờ có hàng chuyên chở (Cargo)
p_carrier = Piece(id="blue_navy_1", piece_type=PieceType.NAVY, owner=Player.BLUE, position=(11, 0), cargo=["blue_infantry_1"])
p_passenger = Piece(id="blue_infantry_1", piece_type=PieceType.INFANTRY, owner=Player.BLUE, position=None, carried_by="blue_navy_1")
assert p_passenger.carried_by == "blue_navy_1"
assert "blue_infantry_1" in p_carrier.cargo
print("PASS: Mô hình hóa quan hệ chuyên chở (Cargo/Carrier) thành công!")

PASS: Đã xác thực đủ 11 loại quân cờ: ['commander', 'infantry', 'militia', 'tank', 'artillery', 'anti_aircraft', 'missile', 'aircraft', 'navy', 'engineer', 'headquarters']
PASS: Khởi tạo quân cờ tiêu chuẩn thành công!
PASS: Mô hình hóa quan hệ chuyên chở (Cargo/Carrier) thành công!


## 2. Khởi tạo Trạng thái Bàn cờ (`GameState` & `create_initial_state`)

Kiểm thử 2 phương thức khởi tạo trạng thái:
1. `GameState()` mặc định (bàn cờ rỗng phục vụ dựng kịch bản unit test).
2. `create_initial_state()` theo bố trí tác chiến chuẩn: 38 quân cờ (19 Đỏ, 19 Xanh), Đỏ đi trước, turn = 1, điểm số ban đầu = 0.

In [3]:
# 2.1. Khởi tạo GameState rỗng
empty_state = GameState()
assert len(empty_state.pieces) == 0, "GameState rỗng không được chứa quân cờ"
assert empty_state.current_player == Player.RED, "Lượt ban đầu mặc định là Đỏ"
assert empty_state.turn_number == 1, "Số lượt khởi đầu phải là 1"
assert empty_state.action_count == 0, "Số hành động khởi đầu phải là 0"
assert empty_state.game_status == GameStatus.ONGOING, "Trạng thái ván đấu ban đầu phải là ONGOING"
assert empty_state.scores == {Player.RED: 0, Player.BLUE: 0}, "Điểm số ban đầu phải là 0-0"
assert empty_state.elapsed_time == {Player.RED: 0.0, Player.BLUE: 0.0}
print("PASS: Khởi tạo GameState rỗng đạt toàn bộ tiêu chí bất biến mặc định.")

# 2.2. Khởi tạo Trạng thái Chuẩn (Standard Initial Setup)
state = create_initial_state()
assert len(state.pieces) == 38, f"Kỳ vọng 38 quân cờ, thực tế có {len(state.pieces)}"

red_pieces = [p for p in state.pieces.values() if p.owner == Player.RED]
blue_pieces = [p for p in state.pieces.values() if p.owner == Player.BLUE]
assert len(red_pieces) == 19, f"Phe Đỏ phải có đúng 19 quân, thực tế có {len(red_pieces)}"
assert len(blue_pieces) == 19, f"Phe Xanh phải có đúng 19 quân, thực tế có {len(blue_pieces)}"

# Kiểm tra tính toàn vẹn cấu trúc ban đầu
state.validate_structure()
print(f"PASS: Khởi tạo trạng thái chuẩn thành công: Tổng cộng {len(state.pieces)} quân (Đỏ: {len(red_pieces)}, Xanh: {len(blue_pieces)}).")
print(f"Lượt đi ban đầu: {state.current_player.value.upper()} | Trạng thái ván: {state.game_status.value}")

PASS: Khởi tạo GameState rỗng đạt toàn bộ tiêu chí bất biến mặc định.
PASS: Khởi tạo trạng thái chuẩn thành công: Tổng cộng 38 quân (Đỏ: 19, Xanh: 19).
Lượt đi ban đầu: RED | Trạng thái ván: ongoing


## 3. Các Phương thức Truy xuất Dữ liệu (Query & Access Methods)

`GameState` cung cấp 3 phương thức truy vấn cốt lõi:
- `get_piece(piece_id)`: Truy xuất quân theo ID trong $O(1)$.
- `get_piece_at(pos)`: Tìm quân chiếm giữ giao điểm $(row, col)$. Ném ngoại lệ `ValueError` nếu tọa độ ngoài biên.
- `get_pieces_by_player(player)`: Lấy danh sách toàn bộ quân hiện còn của phe chỉ định.

In [4]:
state = create_initial_state()

# 3.1. get_piece
cmd_red = state.get_piece("red_commander_1")
assert cmd_red is not None, "Không tìm thấy red_commander_1"
assert cmd_red.piece_type == PieceType.COMMANDER
assert cmd_red.position == (0, 6)

none_piece = state.get_piece("non_existent_id")
assert none_piece is None, "ID không tồn tại phải trả về None"
print("PASS: get_piece() hoạt động chính xác.")

# 3.2. get_piece_at
piece_at_0_6 = state.get_piece_at((0, 6))
assert piece_at_0_6 == cmd_red, "Ô (0, 6) phải là Tư lệnh Đỏ"

empty_spot = state.get_piece_at((5, 5))
assert empty_spot is None, "Ô (5, 5) giữa bàn cờ ban đầu phải trống (None)"

# Ngoại lệ ngoài biên
try:
    state.get_piece_at((-1, 0))
    raise AssertionError("get_piece_at ngoài biên phải ném ValueError")
except ValueError as e:
    assert "Invalid position" in str(e)
    print("PASS: get_piece_at() bắt lỗi tọa độ ngoài biên thành công!")

# 3.3. get_pieces_by_player
red_list = state.get_pieces_by_player(Player.RED)
blue_list = state.get_pieces_by_player(Player.BLUE)
assert len(red_list) == 19
assert len(blue_list) == 19
assert all(p.owner == Player.RED for p in red_list)
assert all(p.owner == Player.BLUE for p in blue_list)
print("PASS: get_pieces_by_player() trả về chính xác danh sách quân của từng phe.")

PASS: get_piece() hoạt động chính xác.
PASS: get_piece_at() bắt lỗi tọa độ ngoài biên thành công!
PASS: get_pieces_by_player() trả về chính xác danh sách quân của từng phe.


## 4. Kiểm soát Ngoại lệ khi Thêm Quân (`add_piece`)

Phương thức `add_piece()` đóng vai trò cổng kiểm soát tính hợp lệ khi xếp quân lên bàn cờ.
Cần kiểm tra toàn bộ 6 rào chắn phòng vệ:
1. `Duplicate piece ID`: Trùng định danh với quân đã có.
2. `Piece must have a position`: Quân thêm vào bàn phải có tọa độ cụ thể.
3. `Cannot directly place carried piece`: Quân đang được chở không được đặt trực tiếp lên bàn.
4. `Cannot directly add a piece with unresolved cargo`: Quân có hàng chở chưa được phân giải.
5. `Invalid position`: Tọa độ vượt ra ngoài biên bàn cờ $12 \times 11$.
6. `Position occupied`: Ô dự định đặt quân đã có quân khác chiếm giữ.

In [5]:
st = GameState()

# Thêm quân đầu tiên thành công
tank = Piece(id="red_tank_1", piece_type=PieceType.TANK, owner=Player.RED, position=(3, 3))
st.add_piece(tank)
assert st.get_piece_at((3, 3)) == tank
print("PASS: Thêm quân hợp lệ đầu tiên thành công.")

# Case 1: Trùng ID
try:
    st.add_piece(Piece(id="red_tank_1", piece_type=PieceType.TANK, owner=Player.RED, position=(4, 4)))
    raise AssertionError("Không bắt được Duplicate piece ID")
except ValueError as e:
    assert "Duplicate piece ID" in str(e)
    print(f"PASS: Case 1 -> {e}")

# Case 2: Quân không có position
try:
    st.add_piece(Piece(id="red_inf_1", piece_type=PieceType.INFANTRY, owner=Player.RED, position=None))
    raise AssertionError("Không bắt được Piece must have a position")
except ValueError as e:
    assert "Piece must have a position" in str(e)
    print(f"PASS: Case 2 -> {e}")

# Case 3: Quân đang được chở
try:
    st.add_piece(Piece(id="red_inf_2", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(1, 1), carried_by="red_navy_1"))
    raise AssertionError("Không bắt được Cannot directly place carried piece")
except ValueError as e:
    assert "Cannot directly place carried piece" in str(e)
    print(f"PASS: Case 3 -> {e}")

# Case 4: Quân có cargo chưa phân giải
try:
    st.add_piece(Piece(id="red_navy_1", piece_type=PieceType.NAVY, owner=Player.RED, position=(0, 0), cargo=["red_inf_3"]))
    raise AssertionError("Không bắt được unresolved cargo")
except ValueError as e:
    assert "unresolved cargo" in str(e)
    print(f"PASS: Case 4 -> {e}")

# Case 5: Tọa độ ngoài biên (12, 0)
try:
    st.add_piece(Piece(id="red_inf_4", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(12, 0)))
    raise AssertionError("Không bắt được Invalid position")
except ValueError as e:
    assert "Invalid position" in str(e)
    print(f"PASS: Case 5 -> {e}")

# Case 6: Ô đã có quân chiếm giữ (3, 3)
try:
    st.add_piece(Piece(id="red_inf_5", piece_type=PieceType.INFANTRY, owner=Player.RED, position=(3, 3)))
    raise AssertionError("Không bắt được Position occupied")
except ValueError as e:
    assert "Position occupied" in str(e)
    print(f"PASS: Case 6 -> {e}")

PASS: Thêm quân hợp lệ đầu tiên thành công.
PASS: Case 1 -> Duplicate piece ID
PASS: Case 2 -> Piece must have a position
PASS: Case 3 -> Cannot directly place carried piece
PASS: Case 4 -> Cannot directly add a piece with unresolved cargo
PASS: Case 5 -> Invalid position
PASS: Case 6 -> Position occupied


## 5. Tính Bất biến & Độc lập của Phép Sao chép Sâu (`copy()` / Deepcopy)

Các thuật toán tìm kiếm cây trò chơi (Minimax, Alpha-Beta, MCTS) phụ thuộc hoàn toàn vào khả năng clone trạng thái:
- Phép `state.copy()` phải thực hiện sao chép sâu (`deepcopy`) toàn bộ cấu trúc: `pieces`, `board`, `scores`, `compound_turn`.
- **Yêu cầu bất biến:** Mọi thao tác di chuyển, xóa quân, tăng điểm trên bản sao **tuyệt đối không làm thay đổi trạng thái gốc**.

In [6]:
state_original = create_initial_state()
state_clone = state_original.copy()

# 5.1. Kiểm tra id bộ nhớ phân tách hoàn toàn
assert id(state_original) != id(state_clone), "Hai GameState phải có ID bộ nhớ khác nhau"
assert id(state_original.pieces) != id(state_clone.pieces), "Dict pieces phải được deepcopy"
assert id(state_original.scores) != id(state_clone.scores), "Dict scores phải được deepcopy"
assert id(state_original.board) != id(state_clone.board), "Board instance phải được deepcopy"
print("PASS: Bản sao có con trỏ bộ nhớ hoàn toàn độc lập với bản gốc.")

# 5.2. Đột biến trên bản sao và kiểm chứng bản gốc bất biến
# Sửa vị trí một quân trên clone
orig_cmd_pos = state_original.pieces["red_commander_1"].position
state_clone.pieces["red_commander_1"].position = (4, 5)

# Thêm điểm trên clone
state_clone.scores[Player.RED] = 50
state_clone.turn_number = 15
state_clone.current_player = Player.BLUE

# Kiểm chứng bản gốc không bị ô nhiễm (Mutation Resistance)
assert state_original.pieces["red_commander_1"].position == orig_cmd_pos, "Vị trí quân trên bản gốc bị thay đổi!"
assert state_original.scores[Player.RED] == 0, "Điểm trên bản gốc bị thay đổi!"
assert state_original.turn_number == 1, "Turn number bản gốc bị thay đổi!"
assert state_original.current_player == Player.RED, "Current player bản gốc bị thay đổi!"

print(f"Vị trí ban đầu bản gốc: {state_original.pieces['red_commander_1'].position}")
print(f"Vị trí sau khi sửa trên clone: {state_clone.pieces['red_commander_1'].position}")
print("PASS: Tính bất biến qua deepcopy được bảo toàn 100%, sẵn sàng cho tìm kiếm đa luồng và đệ quy.")

PASS: Bản sao có con trỏ bộ nhớ hoàn toàn độc lập với bản gốc.
Vị trí ban đầu bản gốc: (0, 6)
Vị trí sau khi sửa trên clone: (4, 5)
PASS: Tính bất biến qua deepcopy được bảo toàn 100%, sẵn sàng cho tìm kiếm đa luồng và đệ quy.


## 6. Bảo vệ Bất biến Cấu trúc & Quan hệ Chuyên chở (`validate_structure()`)

Hàm `validate_structure()` là bức tường bảo vệ tính nhất quán dữ liệu của bàn cờ:
- Quan hệ 2 chiều giữa Phương tiện và Hàng chở: Nếu `A.cargo` chứa `B` thì `B.carried_by` phải bằng `A`, và `B.position` phải là `None`.
- Đồng nhất phe: Phương tiện và Hàng chở phải cùng một phe.
- Phát hiện chu trình chuyên chở lặp vòng (ví dụ: A chở B, B chở A).
- Ràng buộc lượt ghép (`CompoundTurnState`): Phương tiện gốc hợp lệ, số hành động $0 \le \text{actions} \le 3$.

In [7]:
# 6.1. Thiết lập quan hệ chuyên chở hợp lệ
st_valid = GameState()
navy_red = Piece(id="red_navy_1", piece_type=PieceType.NAVY, owner=Player.RED, position=(0, 0), cargo=["red_inf_1"])
inf_red = Piece(id="red_inf_1", piece_type=PieceType.INFANTRY, owner=Player.RED, position=None, carried_by="red_navy_1")
st_valid.pieces["red_navy_1"] = navy_red
st_valid.pieces["red_inf_1"] = inf_red
st_valid.validate_structure()
print("PASS: Cấu trúc chuyên chở hợp lệ vượt qua validate_structure().")

# 6.2. Phát hiện lỗi quân được chở nhưng vẫn có tọa độ trên bàn cờ
st_err1 = st_valid.copy()
st_err1.pieces["red_inf_1"].position = (0, 1)  # Lỗi: vừa được chở vừa đứng trên bàn
try:
    st_err1.validate_structure()
    raise AssertionError("Không phát hiện quân được chở có vị trí!")
except ValueError as e:
    assert "Carried piece red_inf_1 must have no position" in str(e)
    print(f"PASS: Phát hiện lỗi vị trí quân được chở -> {e}")

# 6.3. Phát hiện lỗi chuyên chở khác phe (Carrier owner mismatch)
st_err2 = st_valid.copy()
st_err2.pieces["red_inf_1"].owner = Player.BLUE  # Lỗi: Tàu Đỏ chở lính Xanh
try:
    st_err2.validate_structure()
    raise AssertionError("Không phát hiện chuyên chở khác phe!")
except ValueError as e:
    assert "Carrier owner mismatch" in str(e)
    print(f"PASS: Phát hiện chuyên chở khác phe -> {e}")

# 6.4. Phát hiện chu trình chuyên chở lặp vòng (Circular transport loop)
st_loop = GameState()
pA = Piece(id="pA", piece_type=PieceType.NAVY, owner=Player.RED, position=None, carried_by="pB", cargo=["pB"])
pB = Piece(id="pB", piece_type=PieceType.NAVY, owner=Player.RED, position=None, carried_by="pA", cargo=["pA"])
st_loop.pieces["pA"] = pA
st_loop.pieces["pB"] = pB
try:
    st_loop.validate_structure()
    raise AssertionError("Không phát hiện chu trình chuyên chở lặp vòng!")
except ValueError as e:
    assert "Circular transport relationship" in str(e) or "Carrier not found" in str(e) or "Carried piece" in str(e)
    print(f"PASS: Phát hiện cấu trúc vận tải bất thường -> {e}")

# 6.5. Kiểm tra tính hợp lệ của lượt ghép (CompoundTurnState)
st_comp = create_initial_state()
st_comp.compound_turn = CompoundTurnState(
    root_transport_id="red_navy_1",
    origin=(1, 1),
    actions_used=4  # Lỗi: Vượt quá giới hạn tối đa 3 hành động
)
try:
    st_comp.validate_structure()
    raise AssertionError("Không phát hiện actions_used vượt quá 3!")
except ValueError as e:
    assert "Invalid compound action count" in str(e)
    print(f"PASS: Phát hiện lỗi lượt ghép -> {e}")

PASS: Cấu trúc chuyên chở hợp lệ vượt qua validate_structure().
PASS: Phát hiện lỗi vị trí quân được chở -> Carried piece red_inf_1 must have no position
PASS: Phát hiện chuyên chở khác phe -> Carrier owner mismatch for red_inf_1
PASS: Phát hiện cấu trúc vận tải bất thường -> Circular transport relationship
PASS: Phát hiện lỗi lượt ghép -> Invalid compound action count


## 7. Thống kê Trạng thái & Trực quan hóa Bàn cờ Ban đầu

Bảng phân bổ lực lượng của hai phe trên bàn cờ chuẩn và trực quan hóa ma trận bố phòng 132 giao điểm.

In [8]:
state = create_initial_state()

# 7.1. Thống kê số lượng quân cờ theo 11 binh chủng
rows = []
for pt in PieceType:
    count_red = sum(1 for p in state.pieces.values() if p.owner == Player.RED and p.piece_type == pt)
    count_blue = sum(1 for p in state.pieces.values() if p.owner == Player.BLUE and p.piece_type == pt)
    expected = EXPECTED_COUNTS.get(pt, 0)
    rows.append({
        "Binh chủng": pt.value,
        "Đỏ (Red)": count_red,
        "Xanh (Blue)": count_blue,
        "Kỳ vọng mỗi bên": expected,
        "Khớp chuẩn": count_red == expected and count_blue == expected
    })

df_pieces = pd.DataFrame(rows)
print("=== BẢNG THỐNG KÊ LỰC LƯỢNG 11 LOẠI QUÂN TRÊN BÀN CỜ ===")
display(df_pieces)

# 7.2. Trực quan hóa ma trận bố phòng trên bàn cờ 12x11 (ASCII Map)
piece_symbols = {
    PieceType.COMMANDER: "CMD",
    PieceType.INFANTRY: "INF",
    PieceType.MILITIA: "MIL",
    PieceType.TANK: "TNK",
    PieceType.ARTILLERY: "ART",
    PieceType.ANTI_AIRCRAFT: "AA ",
    PieceType.MISSILE: "MIS",
    PieceType.AIRCRAFT: "AIR",
    PieceType.NAVY: "NAV",
    PieceType.ENGINEER: "ENG",
    PieceType.HEADQUARTERS: "HQ "
}

grid = [[" .  " for _ in range(11)] for _ in range(12)]
for p in state.pieces.values():
    if p.position is not None:
        r, c = p.position
        sym = piece_symbols[p.piece_type]
        side_tag = "R" if p.owner == Player.RED else "B"
        grid[r][c] = f"{side_tag}{sym[:2]}"

header_cols = "   ".join([f"C{c:02d}" for c in range(11)])
divider = "   +" + "---+" * 11
print()
print("=== MA TRẬN BỐ PHÒNG 132 GIAO ĐIỂM (Hàng 11: Phe Xanh | Hàng 0: Phe Đỏ) ===")
print(f"      {header_cols}")
print(divider)
for r in range(11, -1, -1):
    row_content = " | ".join(grid[r])
    print(f"R{r:02d}| {row_content} |")
print(divider)

=== BẢNG THỐNG KÊ LỰC LƯỢNG 11 LOẠI QUÂN TRÊN BÀN CỜ ===
       Binh chủng  Đỏ (Red)  Xanh (Blue)  Kỳ vọng mỗi bên  Khớp chuẩn
0       commander         1            1                1        True
1        infantry         2            2                2        True
2         militia         1            1                1        True
3            tank         2            2                2        True
4       artillery         2            2                2        True
5   anti_aircraft         2            2                2        True
6         missile         1            1                1        True
7        aircraft         2            2                2        True
8            navy         2            2                2        True
9        engineer         2            2                2        True
10   headquarters         2            2                2        True

=== MA TRẬN BỐ PHÒNG 132 GIAO ĐIỂM (Hàng 11: Phe Xanh | Hàng 0: Phe Đỏ) ===
      C00   C01   C02   C0

## 8. Kết luận Kiểm thử Game State

Module `commander_game_engine.game_state` hoàn thành toàn bộ các yêu cầu thiết kế hệ thống:
1. **Tính Toàn vẹn (Integrity):** Mô hình hóa chuẩn xác 11 loại quân, 2 phe đối kháng và quan hệ chuyên chở đa binh chủng.
2. **Tính Phòng vệ (Robustness):** Ngăn chặn 100% các tình huống dữ liệu sai lệch (trùng ID, vượt biên, ô bị chiếm, quan hệ vận tải sai phe/vòng lặp).
3. **Tính Bất biến (Immutability):** Hỗ trợ `copy()` sâu hoàn chỉnh, bảo đảm an toàn dữ liệu tuyệt đối khi các tác tử AI duyệt cây tìm kiếm.
4. **Trạng thái Sẵn sàng:** `GameState` hoàn toàn sẵn sàng tích hợp với bộ sinh nước đi (`move_generator`), bộ luật cờ (`rules`) và bộ điều phối động cơ (`engine`).